In [ ]:
# ===== NB05 | CELL 1: connect Drive and add two folders =====
# WHAT: mount Drive, point to YOUR existing project folder, add folders for predictions and final results.
# WHY:  we keep working inside the folder from notebooks 00-04, so nothing is moved or redone.
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
for d in ['src', 'results/predictions', 'results/final']:
    os.makedirs(f'{PROJECT_DIR}/{d}', exist_ok=True)
print('Processed files found:', sorted(os.listdir(f'{PROJECT_DIR}/data/processed')))
print('CELL 1 DONE')

In [ ]:
# ===== NB05 | CELL 2: write the shared helper file =====
# WHAT: save toxlib.py (metrics, threshold tuning, saving predictions, experiment log) into the src folder.
# WHY:  every later notebook imports it, so we never copy long functions between notebooks.
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
CODE = r'''"""Shared helpers for the NLP-04 toxic comment project. Every notebook imports this file."""
import os, time
import numpy as np, pandas as pd
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             f1_score, precision_score, recall_score)

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
SPLITS = ['val', 'test_internal', 'test_external']


def norm_text(s):
    """Key for spotting identical comments (lowercase, trimmed, single spaces)."""
    return s.astype(str).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()


def load_split(proj, name):
    return pd.read_csv(f'{proj}/data/processed/{name}.csv')


def thr_vec(thr):
    if np.isscalar(thr):
        return np.full(len(LABELS), float(thr))
    if isinstance(thr, dict):
        return np.array([thr[l] for l in LABELS], dtype=float)
    return np.array(thr, dtype=float)


def macro_ap(Y, P):
    """Macro Average Precision (our main selection metric)."""
    return float(np.mean([average_precision_score(Y[:, i], P[:, i]) for i in range(len(LABELS))]))


def tune_thresholds(Y, P, grid=None):
    """Best-F1 threshold per label. VALIDATION data only."""
    grid = np.arange(0.05, 0.951, 0.01) if grid is None else grid
    best = {}
    for i, l in enumerate(LABELS):
        f1s = [f1_score(Y[:, i], P[:, i] >= t, zero_division=0) for t in grid]
        best[l] = round(float(grid[int(np.argmax(f1s))]), 2)
    return best


def per_label_report(Y, P, thr=0.5):
    t = thr_vec(thr)
    rows = []
    for i, l in enumerate(LABELS):
        pred = P[:, i] >= t[i]
        rows.append(dict(label=l, n_pos=int(Y[:, i].sum()), threshold=round(t[i], 2),
                         precision=precision_score(Y[:, i], pred, zero_division=0),
                         recall=recall_score(Y[:, i], pred, zero_division=0),
                         F1=f1_score(Y[:, i], pred, zero_division=0),
                         AP=average_precision_score(Y[:, i], P[:, i]),
                         ROC_AUC=roc_auc_score(Y[:, i], P[:, i])))
    return pd.DataFrame(rows).round(4)


def save_probs(proj, run_id, split, P):
    d = f'{proj}/results/predictions'
    os.makedirs(d, exist_ok=True)
    np.save(f'{d}/{run_id}_{split}.npy', np.asarray(P, dtype='float32'))


def load_probs(proj, run_id, split):
    return np.load(f'{proj}/results/predictions/{run_id}_{split}.npy')


def log_run(proj, run_id, **info):
    """One row per experiment in results/experiment_logs/registry.csv (replaces the row if re-run)."""
    path = f'{proj}/results/experiment_logs/registry.csv'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    row = pd.DataFrame([dict(run_id=run_id, time=time.strftime('%Y-%m-%d %H:%M'), **info)])
    if os.path.exists(path):
        old = pd.read_csv(path)
        row = pd.concat([old[old.run_id != run_id], row], ignore_index=True)
    row.to_csv(path, index=False)
    return row
'''
import os, sys, importlib
os.makedirs(f'{PROJECT_DIR}/src', exist_ok=True)
open(f'{PROJECT_DIR}/src/toxlib.py', 'w').write(CODE)
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
print('Helper written and imported. Labels:', toxlib.LABELS)
print('CELL 2 DONE')

In [ ]:
# ===== NB05 | CELL 3: build the external test set (the labelled Kaggle test) =====
# WHAT: keep only Kaggle test rows that have real labels (-1 means 'not scored'), drop any text also in train.csv.
# WHY:  a second test set from a separate source, with 211 threat comments, so rare-label scores are more reliable.
import sys, importlib
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, norm_text
import pandas as pd, numpy as np
RAW, PROC = f'{PROJECT_DIR}/data/raw', f'{PROJECT_DIR}/data/processed'

train = pd.read_csv(f'{RAW}/train.csv')
train_keys = set(norm_text(train.comment_text))

ext = pd.read_csv(f'{RAW}/test.csv').merge(pd.read_csv(f'{RAW}/test_labels.csv'), on='id')
n_all = len(ext)
ext = ext[(ext[LABELS] != -1).all(axis=1)].copy()
n_lab = len(ext)
ext['key'] = norm_text(ext.comment_text)
overlap = ext.key.isin(train_keys)
ext = ext[~overlap]
ext[['id', 'comment_text'] + LABELS].to_csv(f'{PROC}/test_external.csv', index=False)
print(f'Kaggle test rows: {n_all} | fully labelled (no -1): {n_lab} | '
      f'removed (text also in train.csv): {int(overlap.sum())} | kept: {len(ext)}')
print('Label counts in external test:', ext[LABELS].sum().to_dict())
print('CELL 3 DONE')

In [ ]:
# ===== NB05 | CELL 4: leakage checks + split summary table =====
# WHAT: re-verify that no comment text sits in two different parts, and save the summary table (T06).
# WHY:  one clean proof that train, validation, internal test and external test are fully separate.
import sys, importlib
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, norm_text, load_split
import pandas as pd, numpy as np

names = ['train_full', 'train_sub', 'val', 'test_internal', 'test_external']
d = {n: load_split(PROJECT_DIR, n) for n in names}
keys = {n: set(norm_text(d[n].comment_text)) for n in names}
ids = {n: set(d[n].id) for n in names}

checks = {
    'train_full / val text overlap': len(keys['train_full'] & keys['val']),
    'train_full / test_internal text overlap': len(keys['train_full'] & keys['test_internal']),
    'val / test_internal text overlap': len(keys['val'] & keys['test_internal']),
    'test_external / train_full text overlap': len(keys['test_external'] & keys['train_full']),
    'test_external / val text overlap': len(keys['test_external'] & keys['val']),
    'test_external / test_internal text overlap': len(keys['test_external'] & keys['test_internal']),
    'train_sub rows not in train_full': len(ids['train_sub'] - ids['train_full']),
}
print('Leakage checks (all must be 0):')
for k, v in checks.items():
    print(f'  {k:45s} {v}', '' if v == 0 else '  <-- PROBLEM')

rows = []
for n in names:
    r = {'part': n, 'rows': len(d[n]), 'clean %': round(100 * (d[n][LABELS].sum(axis=1) == 0).mean(), 1)}
    r.update({l: int(d[n][l].sum()) for l in LABELS})
    rows.append(r)
T = pd.DataFrame(rows)
T.to_csv(f'{PROJECT_DIR}/results/tables/T06_split_summary.csv', index=False)
print('\nT06 split summary:'); print(T.to_string(index=False))
print('CELL 4 DONE')